In [1]:
%load_ext autoreload
%autoreload 2
import torch

from src.ctc.config import CTCConfig as C
from src.ctc.model import CTCModel


device = C.get_device()
print(f"Using device: {device}")

checkpoint_path = "../trained_models/ctc_specaugment_70epochs.pt"

checkpoint = torch.load(checkpoint_path, map_location=device)

model = CTCModel().to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

Using device: cpu


CTCModel(
  (conv): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
    (7): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (9): ReLU(inplace=True)
    (10): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (12): ReLU(inplace=True)
    (13): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
    (14):

In [2]:
from src.wordmaker import PHONEME_TO_LETTERS, levenshtein_distance, phonemes_to_text, WLIST1000

In [3]:
from src.ctc.features import wav_path_to_logmel
from src.ctc.dataset import textgrid_to_phone_ids
from src.ctc.metrics import greedy_decode, decode_to_phones, compute_per

wav_path = "../AutorskieDane/AutorskiDataset/id2.wav"
tg_path = "../AutorskieDane/AutorskiDataset/id2.TextGrid"
wav_path = "../slowa_testowe/alibaba.wav"
#tf_path = "../slowa_testowe/"


target_ids = textgrid_to_phone_ids(tg_path, map_sp_to_sil=True)


mel = wav_path_to_logmel(wav_path)  # (F, T)
mel = mel.unsqueeze(0).to(device)  # (1, F, T)

with torch.no_grad():
    logits = model(mel)  # (1, T', C+1)

pred_ids = greedy_decode(logits.cpu())[0]  # list[int]


per = compute_per([pred_ids], [target_ids])
print(f"PER for this utterance: {per:.4f}")

print("Target:     ", decode_to_phones(target_ids))
print("Prediction: ", decode_to_phones(pred_ids))

PER for this utterance: 0.9581
Target:      sil s t u d e n tsj i c e r u n k u sil d o v a d u j oc5 sj e sil j a k sil p o s t e m p o v a tsj sil z d u Z i2 m i i l o sj tsj a m i d a n i2 h sil p o h o dz o n c i2 m i sil z r u Z n i2 h zj r u d e w sil u tS oc5 sj e r u v n~ e S j e i n t e r p r e t o v a tsj sil f S i2 s t k o t o sil v o p a r tsj u o sil p r o g r a m i2 sil i sil a l g o r i2 t m i2 k o m p u t e r o v e sil a sil t a k S e sil v e dz e sil s m a t e m a t i2 c i sil s t a t i2 s t i2 c i sil i e k o n o m j i sil
Prediction:  sil a l j b a b a sil t r u


In [4]:
wav_path = "../slowa_testowe/piesek.wav"


target_ids = textgrid_to_phone_ids(tg_path, map_sp_to_sil=True)


mel = wav_path_to_logmel(wav_path)  # (F, T)
mel = mel.unsqueeze(0).to(device)  # (1, F, T)

with torch.no_grad():
    logits = model(mel)  # (1, T', C+1)

pred_ids = greedy_decode(logits.cpu())[0]  # list[int]


per = compute_per([pred_ids], [target_ids])
print(f"PER for this utterance: {per:.4f}")

print("Target:     ", decode_to_phones(target_ids))
print("Prediction: ", decode_to_phones(pred_ids))

decoded = decode_to_phones(pred_ids)
decoded_text = phonemes_to_text(decoded)
w = [str(ph) for ph in decoded_text.split()]
wtext = phonemes_to_text(w, after_silence=False)
output = WLIST1000[0]
mindist = 1000
for wr in WLIST1000:
    #print(f"Testing word: {wr}")
    dist = levenshtein_distance(wr, w)
    #print(dist)
    if dist < mindist:
        mindist = dist
        output = wr
print("_______________________________________________________________")
print(f"Prediction without correction: {wtext}")
print(f"Best match:  ---- {output} ----- with distance {mindist}")

PER for this utterance: 0.9721
Target:      sil s t u d e n tsj i c e r u n k u sil d o v a d u j oc5 sj e sil j a k sil p o s t e m p o v a tsj sil z d u Z i2 m i i l o sj tsj a m i d a n i2 h sil p o h o dz o n c i2 m i sil z r u Z n i2 h zj r u d e w sil u tS oc5 sj e r u v n~ e S j e i n t e r p r e t o v a tsj sil f S i2 s t k o t o sil v o p a r tsj u o sil p r o g r a m i2 sil i sil a l g o r i2 t m i2 k o m p u t e r o v e sil a sil t a k S e sil v e dz e sil s m a t e m a t i2 c i sil s t a t i2 s t i2 c i sil i e k o n o m j i sil
Prediction:  sil k s e k sil
_______________________________________________________________
Prediction without correction: ksek
Best match:  ---- pasek ----- with distance 3


In [5]:
mel[:, :10].shape

torch.Size([1, 10, 386])

In [6]:
from src.wordmaker import dictionary_extend
null_dict = []
nd = dictionary_extend(null_dict, "../AutorskieDane/AutorskiDataset")


znaleziono 46 plików


In [19]:
from pathlib import Path
from src.wordmaker import parse_words

PATH = Path('../AutorskieDane/AutorskiDataset/Potop1.TextGrid')

with open(PATH, "r", encoding="utf-8") as f:
    wordss = parse_words(f.read())

wav_path = '../AutorskieDane/AutorskiDataset/Potop1.wav'

mel = wav_path_to_logmel(wav_path)

In [17]:
def mel_cut(word_info, mel): #(start_time, end_time, word)
    hop_time = C.FRAME_MS / 1000
    n_start = int(word_info[0] / hop_time)
    n_end = int(word_info[1] / hop_time)
    mel_exact = mel[:,n_start:n_end]
    return mel_exact

def predict_from_exact_mel(target_word, mel_exact, dictionary, model, verbose=False, proba_threshold=0.67, top_phonemes=3):

    
    mel_exact = mel_exact.unsqueeze(0).to(device) 

    with torch.no_grad():
        logits = model(mel_exact)  # (1, T', C+1)
    
    pred_ids = greedy_decode(logits.cpu())[0]  # list[int]

    decoded = decode_to_phones(pred_ids)
    decoded_text = phonemes_to_text(decoded)
    #w = [str(ph) for ph in decoded_text.split()]
    #wtext = phonemes_to_text(w, after_silence=False)
    
    output = dictionary[0]
    mindist = 1000
    for wr in dictionary:
        dist = levenshtein_distance(wr, decoded_text)
        if dist < mindist:
            mindist = dist
            output = wr
    if(verbose):
        print(f'output: {output}')
        print(f'target: {target_word}')
    return output

In [25]:
PATH = Path('../AutorskieDane/AutorskiDataset/lalka3.TextGrid') 
wav_path = '../AutorskieDane/AutorskiDataset/lalka3.wav'
#żeby se przetestować to usstaw tak aby w obu ścieżkach był ten sam plik 

with open(PATH, "r", encoding="utf-8") as f:
    wordss = parse_words(f.read())

mel = wav_path_to_logmel(wav_path)


good = 0 
for w_info in wordss:
    if(w_info[2] != 'sil' and (w_info[1] - w_info[0]) > C.WIN_MS/1000):
        #print(w_info[1] - w_info[0])
        #print(C.FRAME_MS/1000)
        mel_exact = mel_cut(w_info, mel)
        output = predict_from_exact_mel(w_info[2], mel_exact, null_dict, model)
        if output == w_info[2]:
            good += 1
print(good/len(wordss))

0.24489795918367346
